In [0]:
# ================================================================
# NOTEBOOK: nb_silver_orders_initial
# PURPOSE:  One-time full load from Bronze → Silver
# RUN:      ONCE only — never run again after first execution
# ================================================================


from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim, when, upper, to_timestamp
from pyspark.sql.window import Window

BRONZE_PATH = "abfss://source@stshopsensedevhj.dfs.core.windows.net/bronze/orders/"
SILVER_PATH = "abfss://source@stshopsensedevhj.dfs.core.windows.net/silver/orders/"

# Read full Bronze
bronze_df = spark.read.parquet(BRONZE_PATH)
print(f"[BRONZE] Rows read: {bronze_df.count()}")


# Deduplication
dedup_window = Window.partitionBy("OrderID").orderBy(F.desc("LastModifiedDate"))
bronze_df = (
    bronze_df
    .withColumn("_rn", F.row_number().over(dedup_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

# Data Quality
bronze_df = (
    bronze_df
    .filter(col("OrderID").isNotNull())
    .filter(col("CustomerID").isNotNull())
    .filter(col("TotalAmount").isNotNull())
)

# Type casting, cleaning, derived columns
silver_df = (
    bronze_df
    .withColumn("OrderDate",        to_timestamp("OrderDate"))
    .withColumn("ShippedDate",      to_timestamp("ShippedDate"))
    .withColumn("DeliveredDate",    to_timestamp("DeliveredDate"))
    .withColumn("LastModifiedDate", to_timestamp("LastModifiedDate"))
    .withColumn("TotalAmount",      col("TotalAmount").cast("decimal(10,2)"))
    .withColumn("DiscountAmount",   col("DiscountAmount").cast("decimal(10,2)"))
    .withColumn("ShippingCharges",  col("ShippingCharges").cast("decimal(10,2)"))
    .withColumn("OrderStatus",      upper(trim(col("OrderStatus"))))
    .withColumn("PaymentMethod",    upper(trim(col("PaymentMethod"))))
    .withColumn("IsPrimeOrder",     col("IsPrimeOrder") == "TRUE")

    # Derived Columns
    .withColumn("OrderYear", F.year("OrderDate"))
    .withColumn("OrderMonth", F.month("OrderDate"))
    .withColumn("OrderDayOfWeek", F.dayofweek("OrderDate"))
    .withColumn("IsWeekendOrder",   col("OrderDayOfWeek").isin([1,7]))
    .withColumn("IsDelivered",      col("OrderStatus") == "DELIVERED")
    .withColumn("IsCancelled",      col("OrderStatus") == "CANCELLED")
    .withColumn("IsReturned",       col("OrderStatus") == "RETURNED")
    .withColumn("NetAmount",        col("TotalAmount") - col("DiscountAmount") - col("ShippingCharges"))
    .withColumn("DaysToDeliver",    F.when(col("DeliveredDate").isNotNull(),
                                           F.datediff(col("DeliveredDate"), col("OrderDate"))).otherwise(None))
    .withColumn("_silver_load_ts",  F.current_timestamp())
    .withColumn("_source",          F.lit("initial_full_load"))    
    .withColumn("_is_deleted",      F.lit(False))
)

    # Write to silver layer

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("OrderYear", "OrderMonth")
    .save(SILVER_PATH)

)

total = silver_df.filter(col("IsDelivered")).count()
delivered = silver_df.filter(col("IsDelivered")).count()
cancelled = silver_df.filter(col("IsCancelled")).count()

print(f"[OK]Silver orders written:{total} rows")
print(f"    Delivered:{delivered}| Cancelled: {cancelled}")
print(f"    Path:{SILVER_PATH}")
print("[DONE] Run nb_silver_orders_cdc_merge daily from now on.")







[BRONZE] Rows read: 3002
[OK]Silver orders written:1453 rows
    Delivered:1453| Cancelled: 405
    Path:abfss://source@stshopsensedevhj.dfs.core.windows.net/silver/orders/
[DONE] Run nb_silver_orders_cdc_merge daily from now on.
